# K-ABENA — CNN Vision : PyTorch ↔ TensorFlow

**Architecture identique** — ConvNet 3 couches — implémentée dans les deux frameworks.
K-ABENA filtre les **images déjà bien classifiées** (perte ≤ K) avant la backward pass.

```
Forward pass : toutes les images (n)
    ↓
Filtre K-ABENA : ne garder que les m images avec perte > K
    ↓
Backward pass : m ≤ n images actives
```

In [ ]:
# Hyperparamètres communs
K_ABENA    = 0.30   # perte BCE > 0.30 → image difficile (active)
N_ABENA    = 0.40   # conserver 40% des images faciles
BATCH_SIZE = 128
EPOCHS     = 5      # réduit pour démo — augmenter pour production
LR         = 0.01

---
# VERSION PYTORCH — CNN CIFAR-10

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision, torchvision.transforms as T
from kabena_ml.integrations.torch_utils import kabena_filter_torch

# Dataset
tfm    = T.Compose([T.ToTensor(), T.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))])
train  = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=tfm)
loader = torch.utils.data.DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)

# Modèle CNN PyTorch
class CNN_PT(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(128*4*4, 256), nn.ReLU(),
            nn.Dropout(0.5), nn.Linear(256, 10)
        )
    def forward(self, x): return self.head(self.features(x))

model_pt  = CNN_PT()
opt_pt    = torch.optim.SGD(model_pt.parameters(), lr=LR, momentum=0.9, weight_decay=1e-4)

# Boucle PyTorch + K-ABENA
for epoch in range(EPOCHS):
    total_m, total_n, losses_ep = 0, 0, []
    for X_b, y_b in loader:
        losses = F.cross_entropy(model_pt(X_b), y_b, reduction='none')  # ← 'none'
        mask   = kabena_filter_torch(losses, K=K_ABENA, N=N_ABENA)      # ← +1 ligne
        if not mask.any(): continue
        L_KA = losses[mask].mean()
        opt_pt.zero_grad(); L_KA.backward(); opt_pt.step()
        total_m += mask.sum().item(); total_n += len(y_b)
        losses_ep.append(L_KA.item())
    import numpy as np
    gain = round((1 - total_m/total_n)*100)
    print(f"[PT] Epoch {epoch+1}/{EPOCHS} | loss={np.mean(losses_ep):.4f} | gain={gain}%")

---
# VERSION TENSORFLOW — CNN CIFAR-10

In [ ]:
import tensorflow as tf
from kabena_ml.integrations.tf_utils import KabenaCallback, KabenaTFTrainer
from kabena_ml import KabenaConfig

# Dataset CIFAR-10 via TF Datasets
(X_train_tf, y_train_tf), _ = tf.keras.datasets.cifar10.load_data()
X_train_tf = X_train_tf.astype('float32') / 127.5 - 1.0  # normalisation identique à PyTorch
y_train_tf = y_train_tf.squeeze()

# Modèle CNN TensorFlow (même architecture que CNN_PT)
model_tf = tf.keras.Sequential([
    # Bloc 1
    tf.keras.layers.Conv2D(32, 3, padding='same', input_shape=(32,32,3)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.MaxPool2D(2),
    # Bloc 2
    tf.keras.layers.Conv2D(64, 3, padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.MaxPool2D(2),
    # Bloc 3
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPool2D(2),
    # Tête
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10),
])

model_tf.compile(
    optimizer=tf.keras.optimizers.SGD(LR, momentum=0.9, weight_decay=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# AVANT K-ABENA : model_tf.fit(X_train_tf, y_train_tf, ...)
# APRÈS K-ABENA : +1 callback
model_tf.fit(
    X_train_tf, y_train_tf,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[KabenaCallback(K=K_ABENA, N=N_ABENA, verbose=True)],  # ← +1 ligne
    verbose=0,
)

## Table de correspondance CNN — PyTorch ↔ TensorFlow

| Composant | PyTorch | TensorFlow |
|-----------|---------|------------|
| Conv2D | `nn.Conv2d(in, out, k, padding=1)` | `tf.keras.layers.Conv2D(out, k, padding='same')` |
| BatchNorm | `nn.BatchNorm2d(out)` | `tf.keras.layers.BatchNormalization()` |
| MaxPool | `nn.MaxPool2d(2)` | `tf.keras.layers.MaxPool2D(2)` |
| Dropout | `nn.Dropout(0.5)` | `tf.keras.layers.Dropout(0.5)` |
| Forward | `model(x)` | `model(x, training=True/False)` |
| **K-ABENA** | `kabena_filter_torch(losses, K, N)` | `KabenaCallback(K, N)` |
| Loss individuelle | `F.cross_entropy(..., reduction='none')` | automatique dans callback |

> **Note format images** : PyTorch attend `(N, C, H, W)`, TensorFlow attend `(N, H, W, C)`.
> K-ABENA est agnostique au format — il ne voit que la perte scalaire par image.